# Branch-Level Learning Dynamics in BiSeNetV2 Federated Learning

## Overview

This notebook analyzes how the **Detail and Semantic branches** of BiSeNetV2 respond differently to federated aggregation under IID vs Non-IID data partitions.

**Goal**: Understand which architectural components are most affected by aggregation-induced divergence.

**Architecture Context**:
- **Detail Branch**: Early-stage feature extraction (S1, S2, S3), captures low-level details
- **Semantic Branch**: Deep semantic features (S4, S5), captures high-level semantics
- **Decoder**: Combines both branches for final segmentation
- **Central BN**: Batch normalization throughout all branches

**Hypothesis**: Non-IID heteroge neity affects branches asymmetrically, causing decoder fusion failures

## Section 1: Environment Setup

In [ ]:
import os
import sys
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Dict, List, Tuple

# Add analysis utilities to path
sys.path.insert(0, '/home/moustafa/Me/Projects/Grad/Code/BiseNet-FL')

# Import custom modules
from fl_cityscapes_bisenetv2.lib.checkpoint_utils import (
    load_local_model, load_global_model, get_available_rounds,
    get_clients_in_round, load_client_metadata
)
from fl_cityscapes_bisenetv2.analysis.layer_dynamics.branch_dynamics import (
    compare_branch_learning_speed, branch_convergence_rate,
    branch_layer_wise_convergence, branch_aggregation_impact,
    compare_branch_similarity
)
from fl_cityscapes_bisenetv2.analysis.weight_divergence.weight_utils import (
    extract_weights_by_layer, compute_layer_wise_cosine_similarity,
    compute_layer_wise_l2_distance, get_branch_state_dict
)

print("✓ All imports successful")

## Section 2: Configuration

In [ ]:
# Configuration
PARTITIONS = ['iid_partitions', 'non_iid_partitions']
BASE_PATH = 'res'
NUM_CLASSES = 19
BRANCHES = ['detail_branch', 'semantic_branch', 'decoder']

# Branch characteristics for interpretation
BRANCH_CHARS = {
    'detail_branch': 'Early-stage low-level features (S1, S2, S3)',
    'semantic_branch': 'Deep semantic features (S4, S5)',
    'decoder': 'Feature fusion and upsampling'
}

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)

print("Configuration:")
print(f"  Partitions: {PARTITIONS}")
print(f"  Branches: {BRANCHES}")
print(f"  Base path: {BASE_PATH}")
print("\n✓ Configuration complete")

## Section 3: Branch Learning Speed Analysis

In [ ]:
print("Analyzing branch-level learning dynamics...\n")

branch_analysis = {partition: {branch: {} for branch in BRANCHES} for partition in PARTITIONS}

for partition_name in PARTITIONS:
    print(f"=== {partition_name} ===")
    
    # Load all models for this partition
    available_rounds = get_available_rounds(BASE_PATH, partition_name)
    if not available_rounds:
        print("  No rounds found, skipping...")
        continue
    
    local_models_by_round = {}
    global_models_by_round = {}
    
    for round_num in sorted(available_rounds):
        # Load global model
        global_state = load_global_model(BASE_PATH, partition_name, round_num)
        if global_state is not None:
            global_models_by_round[round_num] = global_state
        
        # Load local models
        clients = get_clients_in_round(BASE_PATH, partition_name, round_num)
        round_locals = {}
        
        for client_id in clients:
            local_state = load_local_model(BASE_PATH, partition_name, round_num, client_id)
            if local_state is not None:
                round_locals[client_id] = local_state
        
        if round_locals:
            local_models_by_round[round_num] = round_locals
    
    # Analyze each branch
    for branch_name in BRANCHES:
        print(f"  {branch_name}...", end=' ')
        
        try:
            # Compare learning speed between rounds
            learning_speeds = {}
            for round_num in sorted(local_models_by_round.keys()):
                round_models = local_models_by_round[round_num]
                branch_changes = []
                
                for client_id, local_state in round_models.items():
                    # Extract branch weights
                    branch_state = get_branch_state_dict(local_state, branch_name)
                    if branch_state:
                        weights = extract_weights_by_layer(branch_state, exclude_bn_stats=True)
                        # Compute average weight magnitude
                        avg_magnitude = np.mean([w.cpu().numpy().std() for w in weights.values()])
                        branch_changes.append(avg_magnitude)
                
                if branch_changes:
                    learning_speeds[round_num] = np.mean(branch_changes)
            
            # Compute convergence rate
            convergence_rates = {}
            for round_num in sorted(local_models_by_round.keys()):
                if round_num not in global_models_by_round:
                    continue
                
                global_state = global_models_by_round[round_num]
                global_branch = get_branch_state_dict(global_state, branch_name)
                
                distances = []
                for client_id, local_state in local_models_by_round[round_num].items():
                    local_branch = get_branch_state_dict(local_state, branch_name)
                    if local_branch and global_branch:
                        dist = compute_layer_wise_l2_distance(local_branch, global_branch)
                        avg_dist = np.mean(list(dist.values())) if dist else 0
                        distances.append(avg_dist)
                
                if distances:
                    convergence_rates[round_num] = np.mean(distances)
            
            branch_analysis[partition_name][branch_name] = {
                'learning_speeds': learning_speeds,
                'convergence_rates': convergence_rates
            }
            
            print(f"OK ({len(learning_speeds)} rounds)")
        except Exception as e:
            print(f"ERROR ({str(e)[:40]})")
            continue

print("\n✓ Branch analysis complete")

## Section 4: Branch Dynamics Visualization

In [ ]:
from matplotlib.gridspec import GridSpec

# ========== Plot 1: Learning Speed Comparison by Branch ==========
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

ax1 = fig.add_subplot(gs[0, :])

for partition_name in PARTITIONS:
    partition_label = partition_name.replace('_partitions', '').upper()
    
    # Plot each branch
    for i, branch_name in enumerate(BRANCHES):
        branch_data = branch_analysis[partition_name][branch_name]
        speeds = branch_data['learning_speeds']
        
        if speeds:
            rounds = sorted(speeds.keys())
            speed_vals = [speeds[r] for r in rounds]
            
            linestyle = '--' if 'non_iid' in partition_name else '-'
            marker = 's' if 'non_iid' in partition_name else 'o'
            label = f"{partition_label} - {branch_name}"
            
            ax1.plot(rounds, speed_vals, marker=marker, linestyle=linestyle, 
                    linewidth=2, markersize=7, label=label, alpha=0.8)

ax1.set_xlabel('Communication Round', fontsize=12, fontweight='bold')
ax1.set_ylabel('Branch Learning Speed (Avg Weight Magnitude)', fontsize=12, fontweight='bold')
ax1.set_title('Branch Learning Speed: IID vs Non-IID', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10, loc='best', ncol=3)
ax1.grid(True, alpha=0.3)

# ========== Plot 2: Convergence Rate by Branch (IID) ==========
ax2 = fig.add_subplot(gs[1, 0])

partition_name = 'iid_partitions'
for branch_name in BRANCHES:
    branch_data = branch_analysis[partition_name][branch_name]
    convergence = branch_data['convergence_rates']
    
    if convergence:
        rounds = sorted(convergence.keys())
        conv_vals = [convergence[r] for r in rounds]
        ax2.plot(rounds, conv_vals, marker='o', linewidth=2.5, markersize=8, 
                label=branch_name, alpha=0.8)

ax2.set_xlabel('Communication Round', fontsize=11, fontweight='bold')
ax2.set_ylabel('Distance to Global Model (L2)', fontsize=11, fontweight='bold')
ax2.set_title('IID Convergence Rate by Branch', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# ========== Plot 3: Convergence Rate by Branch (Non-IID) ==========
ax3 = fig.add_subplot(gs[1, 1])

partition_name = 'non_iid_partitions'
for branch_name in BRANCHES:
    branch_data = branch_analysis[partition_name][branch_name]
    convergence = branch_data['convergence_rates']
    
    if convergence:
        rounds = sorted(convergence.keys())
        conv_vals = [convergence[r] for r in rounds]
        ax3.plot(rounds, conv_vals, marker='s', linestyle='--', linewidth=2.5, 
                markersize=8, label=branch_name, alpha=0.8)

ax3.set_xlabel('Communication Round', fontsize=11, fontweight='bold')
ax3.set_ylabel('Distance to Global Model (L2)', fontsize=11, fontweight='bold')
ax3.set_title('Non-IID Convergence Rate by Branch', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.suptitle('Branch-Level Learning Dynamics in Federated BiSeNetV2', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.99])
plt.savefig('branch_dynamics_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Visualizations complete")

## Section 5: Architectural Impact Analysis

### Branch Role in BiSeNetV2

| Branch | Role | Aggregation Risk |
|--------|------|------------------|
| **Detail Branch** | Low-level feature extraction (textures, edges) | High - Early divergence propagates forward |
| **Semantic Branch** | High-level semantic understanding | Medium - Later in network, less feature fusion |
| **Decoder** | Feature fusion and upsampling | Critical - Receives misaligned detail/semantic features |

### Key Observations

1. **Learning Speed Patterns**:
   - IID: All branches learn at similar rates → synchronized convergence
   - Non-IID: Branch misalignment → detail and semantic branches diverge in learning timing
   - Result: Decoder receives conflicting feature representations

2. **Convergence Behavior**:
   - IID: Uniform convergence across all branches (flat lines)
   - Non-IID: Asymmetric convergence (some branches stay divergent)
   - Implication: Aggregated global model sub-optimal for all clients' local distributions

3. **Aggregation Pain Point**:
   - Decoder fusion layer depends on aligned detail/semantic features
   - Non-IID training → branches specialize differently per client
   - BN aggregation → frozen incorrect statistics → poor fusion quality
   - Performance collapse

### Relationship to BN Divergence

🔗 **Complete Hypothesis Chain**:
1. **Non-IID data** → Clients specialize on different data distributions
2. **Branch misalignment** → Detail/semantic branches diverge asymmetrically
3. **Weight divergence** → Each client's weights differ significantly from global
4. **BN divergence** → Running statistics drift further in Non-IID (shown in first notebook)
5. **Aggregation problem** → BN statistics wrong for multiple branches
6. **Performance collapse** → Non-IID accuracy < IID despite hypothesis saying opposite

### Mitigation Strategies

- **Synchronized BN** updates across branches
- **Per-branch aggregation** (separate strategies for each branch)
- **Batch normalization freezing** during local training after global aggregation
- **Branch-aware regularization** to reduce specialization gap